# 🤖 04 — Predictive Modeling
## Bank Marketing Campaign

**Objective:** Build and compare classification models to predict client subscription. Evaluate using appropriate metrics for imbalanced data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import shap
import sys
sys.path.append('..')
from src.utils import load_data, evaluate_model, plot_roc_curves, plot_confusion_matrix

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = load_data('../data/bank-additional-full.csv')
df['y'] = (df['y'] == 'yes').astype(int)
print(f'Target distribution:\n{df["y"].value_counts(normalize=True)}')

## 1. Data Preprocessing

In [ ]:
# Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Split features and target
X = df_encoded.drop('y', axis=1)
y = df_encoded['y']

# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features
scaler = StandardScaler()
numerical_cols = df.select_dtypes(include=[np.number]).columns.drop('y').tolist()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train target balance: {y_train.mean():.3f}')
print(f'Test target balance: {y_test.mean():.3f}')

## 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
metrics_lr = evaluate_model(lr, X_test, y_test, 'Logistic Regression')
plot_confusion_matrix(y_test, lr.predict(X_test), 'Logistic Regression')

In [ ]:
# Coefficients interpretation (odds ratios)
coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': lr.coef_[0], 'Odds Ratio': np.exp(lr.coef_[0])})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)
coef_df.head(15)

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
metrics_rf = evaluate_model(rf, X_test, y_test, 'Random Forest')
plot_confusion_matrix(y_test, rf.predict(X_test), 'Random Forest')

In [ ]:
# Feature importance
importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_})
importance_df = importance_df.sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
importance_df.plot(x='Feature', y='Importance', kind='barh', ax=ax, color='#2ecc71', legend=False)
ax.set_title('Random Forest — Top 15 Feature Importances')
plt.tight_layout()
plt.savefig('../figures/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
metrics_xgb = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
plot_confusion_matrix(y_test, xgb_model.predict(X_test), 'XGBoost')

## 5. Model Comparison

In [ ]:
# ROC Curves
models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb_model
}
plot_roc_curves(models, X_test, y_test)

In [ ]:
# Summary table
results = pd.DataFrame([metrics_lr, metrics_rf, metrics_xgb])
results = results.set_index('Model').round(4)
results.style.highlight_max(axis=0, color='lightgreen')

## 6. SHAP Explainability (Best Model)

In [ ]:
# SHAP values for the best model
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cross-Validation (Robustness Check)

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    print(f'{name}: ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})')

## Conclusions & Recommendations

**Document your conclusions here:**

1. **Best model**: ...
2. **Key drivers of subscription**: ...
3. **Business recommendation**: ...